# Lab 3\. From Dataset to Batches: Building and Configuring a DataLoader

## Overview

In Lab 2 you loaded the Imagenette dataset and reached individual samples with **train\_data\[i\]**, each one an **(image, label)** pair. A model does not learn from single samples handed over one at a time. It learns from small groups of samples, called batches, drawn in a fresh random order on every pass through the data. The tool that turns a dataset into a stream of batches is the DataLoader, and this lab is where you build and configure one.

You will wrap the training dataset from Lab 2 in a DataLoader, pull a batch out, and read its shape so the idea of a batched tensor becomes concrete. You will then change the settings that matter most and watch what they do: different batch sizes produce different numbers of batches per pass, and turning shuffling on or off changes whether the batches come out mixed or in the dataset's stored order. These are the decisions you make every time you set up training, so seeing their effects directly is what lets you choose them on purpose later.

One practical point shapes the whole lab. A DataLoader builds a batch by stacking samples into a single tensor, and stacking only works when every image is already a tensor of the same size. The Imagenette images from Lab 2 are variable-sized PIL images, which cannot be stacked as they are. To make batching work, you apply a small transform when you load the data: convert each image to a tensor and resize it to a fixed shape. Transforms are the full subject of Chapter 4; here you use only the minimum needed so the DataLoader has uniform samples to batch.

## Learning Objectives

By the end of this lab, you should be able to:

* Create a DataLoader over the training dataset from Lab 2 with a specified **batch\_size**, **shuffle**, and **drop\_last**.

* Iterate over a DataLoader and describe the shape and structure of each batch of images and labels.

* Determine how many batches a DataLoader produces in one pass, and explain how **drop\_last** changes that count.

* Compare DataLoaders configured with different batch sizes and relate batch size to the number of updates per pass.

* Demonstrate the effect of shuffling by comparing the first batch across two passes of a shuffled and an unshuffled DataLoader.

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:

* A notebook environment: either a Google account for Google Colab, or a local installation of Jupyter Notebook. Every step in this lab runs on a CPU; no GPU is required.

* Python 3.12 or later. On Colab this is already provided. For a local environment, confirm your version with python \--version.

* PyTorch 2.12.1 and TorchVision 0.17 or later (0.27 preferred), installed with pip in the previous lab. If you are on Colab, these may already be present. The Imagenette dataset requires torchvision 0.17 or newer.

* A web browser (Chrome, Firefox, or Edge are recommended for Colab).

>

## Estimated Time

Approximately 30–40 minutes.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new **.ipynb** in local Jupyter. Download the chapter 3 lab notebook from the [LFD2001-resources GitHub repository](https://github.com/lftraining/LFD2001-resources.git) from the **LFD2001-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 3 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the **Sample Output** or **Output** shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

Commands in this lab use the notebook form, where a line beginning with **\!** runs a shell command from inside a notebook cell (for example, **\!pip install ...**). If you run these in a local terminal instead of a notebook cell, drop the leading **\!**.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The steps in this section only prepare the environment. They are not the learning content of the lab, and on a managed course platform some of them may already be done for you. Run them once at the top of your notebook, then move on to Step 1\.

### Setup 1\. Import the tools

Bring in the pieces this lab uses: **torch** for tensors, the **Imagenette** dataset class, the **v2** transforms module (used here only for the minimal preview transform), and the **DataLoader** class.


In [ ]:
import torch
from torchvision.datasets import Imagenette
from torchvision.transforms import v2
from torch.utils.data import DataLoader


The **DataLoader** class lives in **torch.utils.data**, the same module as the **Dataset** you used in Lab 2\.

### Setup 2\. Define a minimal transform so images can be batched

Before the data can be batched, each image has to become a tensor of a fixed size. Define a small transform pipeline that converts a PIL image to a tensor, resizes it to a uniform 160×160, and converts it to a float type. This is a preview of Chapter 4; you use only these three conversion steps here, and nothing else.


In [ ]:
preview_transform = v2.Compose([
    v2.ToImage(),                              # PIL image -> image tensor [C, H, W]
    v2.Resize((160, 160), antialias=True),     # force a uniform 160x160 size
    v2.ToDtype(torch.float32, scale=True),     # cast to float32 and scale pixel values to [0.0, 1.0]
])


Each step earns its place. **ToImage** turns the PIL image into a tensor, which is the only form the DataLoader can stack. **Resize((160, 160))** forces every image to the same height and width, so images that started at different sizes can go into one stacked tensor. **ToDtype(torch.float32, scale=True)** casts the pixel values to a float type and scales them from the 0–255 range into 0.0–1.0, the numeric form a model expects.

### Setup 3\. Load the training and validation splits with the transform

Recreate the two dataset objects from Lab 2, this time passing the transform so every sample comes back as a uniform tensor. Use the same names as Lab 2, **train\_data** and **val\_data**, and the same **data/** root.


In [ ]:
train_data = Imagenette(root="data", split="train", size="160px",
                        download=True, transform=preview_transform)
val_data = Imagenette(root="data", split="val", size="160px",
                      download=True, transform=preview_transform)

print(len(train_data), "training samples")
print(len(val_data), "validation samples")


**Output:**

```
9469 training samples
3925 validation samples
```

The counts match Lab 2, because the transform changes the form of each sample, not how many there are.

>

## Section 1\. Creating and Iterating Over a DataLoader

With uniform tensor samples in place, you can build a DataLoader and see what it delivers. This section creates one DataLoader, reads a single batch, and counts how many batches a full pass produces.

### Step 1\. Create a DataLoader over the training data

Wrap **train\_data** in a DataLoader. Set the batch size to 32, turn shuffling on, and drop the last incomplete batch so every batch is exactly the same size.


In [ ]:
train_loader = DataLoader(
    train_data,
    batch_size=32,
    shuffle=True,
    drop_last=True,
)


Each argument is a decision. **batch\_size=32** puts 32 samples in each batch. **shuffle=True** reshuffles the sample order before every pass, so the batches are mixed rather than in stored order. **drop\_last=True** discards the final batch when the dataset does not divide evenly into full batches, which keeps every batch at 32 samples.

### Step 2\. Pull out one batch and read its shape

A DataLoader is iterable, so you can turn it into an iterator with **iter(...)** and take the first batch with **next(...)**. Each batch is a pair: a tensor of stacked images and a tensor of their labels.


In [ ]:
images, labels = next(iter(train_loader))

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("images dtype:", images.dtype)
print("pixel value range:", images.min().item(), "to", images.max().item())


**Sample Output:**

```
images shape: torch.Size([32, 3, 160, 160])
labels shape: torch.Size([32])
images dtype: torch.float32
pixel value range: 0.0 to 1.0
```

Read the image shape from left to right. The **32** is the batch size, the number of images stacked together. The **3** is the color channels (red, green, and blue). The **160, 160** is the height and width in pixels, uniform because of the resize. The labels are a single dimension of length 32, one integer label per image, lined up in the same order as the images. The dtype and value range confirm the transform did its job: the images are float tensors with values in 0.0–1.0, which is what made stacking possible.

> **Note:** If you skipped the transform in setup and loaded the raw dataset instead, this step would raise an error when the DataLoader tried to stack variable-sized PIL images. The uniform tensor shape you see here is exactly what the transform made possible.

### Step 3\. Count the batches in one pass

Because the DataLoader knows the dataset length and the batch size, it can report how many batches a full pass produces. Ask for it with **len(...)**.


In [ ]:
print("batches per pass:", len(train_loader))


**Output:**

```
batches per pass: 295
```

The training split has 9,469 samples. Divided by a batch size of 32, that is 295 full batches with 29 samples left over. Because **drop\_last=True**, those 29 trailing samples are dropped for the pass and the count is 295\. If you set **drop\_last=False**, the count would be 296, and the last batch would hold only 29 images.

### Step 4\. Iterate over the first few batches

A single batch is useful for inspection, but training walks over every batch in a loop. Iterate over the DataLoader and stop after a few batches to confirm each one has the shape you expect. The **break** keeps the loop short so you are not waiting on all 295 batches.


In [ ]:
for batch_number, (images, labels) in enumerate(train_loader):
    print(f"batch {batch_number}: images {tuple(images.shape)}, labels {tuple(labels.shape)}")
    if batch_number == 2:
        break


**Sample Output:**

```
batch 0: images (32, 3, 160, 160), labels (32,)
batch 1: images (32, 3, 160, 160), labels (32,)
batch 2: images (32, 3, 160, 160), labels (32,)
```

Every batch has the same shape, which is what **drop\_last=True** guarantees. This loop is the shape a training loop takes: its inner loop walks the DataLoader one batch at a time, and each **images, labels** pair is what the model would learn from at that step.

## Section 2\. What Batch Size Does

Batch size affects more than memory. It decides how many batches a pass produces, and because a model updates once per batch, it decides how many updates happen per pass. This section builds DataLoaders at several batch sizes over the same data and compares their batch counts.

### Step 5\. Build DataLoaders at three batch sizes and count their batches

Create three DataLoaders over the same **train\_data**, at batch sizes 16, 64, and 256, and print how many batches each produces. Keep **drop\_last=True** so the comparison is consistent with Section 1\.


In [ ]:
for size in [16, 64, 256]:
    loader = DataLoader(train_data, batch_size=size, shuffle=True, drop_last=True)
    print(f"batch_size={size:>3}  ->  {len(loader)} batches per pass")


**Output:**

```
batch_size= 16  ->  591 batches per pass
batch_size= 64  ->  147 batches per pass
batch_size=256  ->  36 batches per pass
```

The same 9,469 samples produce very different batch counts. A batch size of 16 gives 591 batches, so the model would update 591 times in one pass. A batch size of 256 gives only 36 batches, so 36 updates in the same pass. Smaller batches mean more, smaller updates; larger batches mean fewer, larger ones.

### Step 6\. Record what changes with batch size

No code is needed for this step. Using the counts you just produced and the reasoning from the chapter, write a short note in a text (Markdown) cell answering these questions:

- How does the number of updates per pass change as the batch size grows?  
- Which batch size would give the noisiest updates, and which the smoothest? Why?  
- Which batch size would use the most memory at once? Why?

Comparing the counts to the ideas from the chapter is the point of this section: batch size is a training decision with consequences you can now see in the numbers.

>

## Section 3\. Why Shuffling Matters

Imagenette is stored with all images of one class together before the next class begins. With shuffling off, a DataLoader hands the model long runs of a single class; with shuffling on, each batch mixes classes and each pass draws a fresh order. This section makes that difference visible by comparing the first batch across two passes.

### Step 7\. Create a shuffled and an unshuffled DataLoader

Build two DataLoaders over the same **train\_data**, identical except for the **shuffle** setting. Leave **drop\_last** at its default here, since you are only reading the first batch.


In [ ]:
shuffled_loader = DataLoader(train_data, batch_size=32, shuffle=True)
ordered_loader = DataLoader(train_data, batch_size=32, shuffle=False)


**shuffled\_loader** reshuffles before each pass; **ordered\_loader** always walks the data in stored order.

### Step 8\. Compare the first batch across two passes

Look at the labels in the first batch of each loader, twice. A new **iter(...)** starts a new pass, so calling it twice on the same loader shows whether the first batch changes between passes. Print the first eight labels each time to keep the output short.


In [ ]:
def first_batch_labels(loader):
    images, labels = next(iter(loader))
    return labels[:8].tolist()

print("shuffled, pass 1:", first_batch_labels(shuffled_loader))
print("shuffled, pass 2:", first_batch_labels(shuffled_loader))
print("ordered,  pass 1:", first_batch_labels(ordered_loader))
print("ordered,  pass 2:", first_batch_labels(ordered_loader))


**Sample Output** (the shuffled values will differ for you; the ordered values will not):

```
shuffled, pass 1: [4, 7, 0, 9, 2, 5, 1, 8]
shuffled, pass 2: [6, 1, 3, 0, 7, 2, 9, 4]
ordered,  pass 1: [0, 0, 0, 0, 0, 0, 0, 0]
ordered,  pass 2: [0, 0, 0, 0, 0, 0, 0, 0]
```

The shuffled loader gives a different, mixed set of labels on each pass, because it draws a new random order every time. The ordered loader gives the same labels both times, and they are all **0**. Label **0** is tench, the first class in the dataset's stored order (established in Lab 2), so the unshuffled loader's first batch is nothing but tench images on every pass. A model trained on that order would see a long run of one class before any other, which is the pattern shuffling exists to break.

### Step 9\. Confirm where shuffling belongs

No code is needed here. In a text cell, note why the training loader in Section 1 used **shuffle=True** while an evaluation loader over **val\_data** would use **shuffle=False**.

## Cleanup

This lab does not create anything that must be torn down, and part of what it produces may be kept. There are no processes to stop and no services to remove.

**The `data/` directory** holds the downloaded Imagenette files. On a local machine, keeping it means the next chapter's load is instant instead of downloading again. If you need to reclaim disk space, you may delete **data/imagenette2-160/**, but the dataset will download again the next time you load it.  
>   
On Google Colab the runtime is temporary. When your Colab session ends, the **data/** directory and all the objects are cleared. This is expected. In the next chapter you simply re-run the setup cells, which downloads the data again for the new session.

## Summary

You turned the Lab 2 dataset into batches. You built a DataLoader over **train\_data**, read a batch and saw its **\[32, 3, 160, 160\]** image tensor and **\[32\]** label tensor, and counted how many batches a pass produces under **drop\_last=True**. You then changed the two settings that matter most and watched their effects: different batch sizes produced 591, 147, and 36 batches per pass, and comparing a shuffled loader against an unshuffled one showed the shuffled batches changing every pass while the unshuffled first batch stayed a solid run of the first class, tench.

The one requirement that made all of this possible was uniform tensor samples, which you produced with a minimal three-step transform. That transform was a preview. In Chapter 4 you replace it with a full pipeline that also scales, normalizes, and augments the images, and you attach separate pipelines to the training and validation splits before feeding them to DataLoaders like the ones you built here. The dataset in **data/** and this loading pattern are what you carry into that work.  
